In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')
class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"너는 사용자의 질문에 친철하고 정확하게 답변하는 시스템 입니다., Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,            
        )
        return response.choices[0].message.content
llm = OpenAILLM()    

In [2]:
import json
from  textwrap import dedent  #  들여쓰기 indent를 제거
query = '어제 삼성전자 주식종가 를 조회하고 해당 종가와 비슷한종목을 뉴스에서 검색해서 요약하고 오늘날씨에 맞는 외출복을 추천해줘'
prompt = dedent(f"""
사용자의 질문 의도를 분석하세요.

질문에 포함된 의도가 여러 개이면
반드시 모든 의도를 각각 분리하여 출력하세요.

예를 들어:
- 주식 + 뉴스 + 날씨
- 뉴스 + 검색
- 계산 + 주식

처럼 복합 질문이면
반드시 여러 개의 JSON 객체를 배열로 출력해야 합니다.

question_type 별로 tool_query를 생성하세요.
tool_query는 반드시 키워드중심으로 생성하세요 llm에 전달하는 문구가 아님을 명심해서 추론을하지말고 검색용 키워드로 매칭해주세요    
뉴스는 api검색할수 있는 키워드중심으로,
주식은 종목명만 추출하세요             
날씨도 검색용 키워드로 추출하세요                               

지원 가능한 question_type 예시:
- 날씨
- 뉴스
- 주식
- 검색
- 계산
- 지도

[중요 규칙]
- 질문에 포함된 모든 의도를 누락 없이 출력
- 반드시 JSON 배열(Array) 형식으로 출력
- 하나만 출력 금지
- 설명문 금지
- markdown 금지
- ```json 금지

[예시 입력]
어제 삼성전자 종가 알려주고 관련 뉴스와 오늘 날씨도 알려줘

[예시 출력]
[
    {{
        "question_type": "주식",
        "tool_query": "삼성전자",
        "reason": "삼성전자 종가 요청"
    }},
    {{
        "question_type": "뉴스",
        "tool_query": "삼성전자 관련 최근 뉴스 검색",
        "reason": "관련 뉴스 요청"
    }},
    {{
        "question_type": "날씨",
        "tool_query": "오늘 날씨 조회",
        "reason": "날씨 요청"
    }}
]

[질문]
{query}

[출력]
반드시 유효한 JSON 배열만 출력하세요.
""")

result = llm.generate(prompt)
json_result = json.loads(result)
json_result

[{'question_type': '주식',
  'tool_query': '삼성전자',
  'reason': '어제 삼성전자 주식 종가 조회 요청'},
 {'question_type': '뉴스',
  'tool_query': '삼성전자 종가 유사 종목 뉴스 요약',
  'reason': '해당 종가와 비슷한 종목을 뉴스에서 검색해 요약 요청'},
 {'question_type': '날씨',
  'tool_query': '오늘 날씨 외출복 추천',
  'reason': '오늘 날씨에 맞는 외출복 추천 요청'}]

In [3]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECETET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(query)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()    
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)        
        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [7]:
for row in json_result:
    if row.get('question_type') == '뉴스':  
        print('검색어',row.get('tool_query'))
        print(search_naver_news(row.get('tool_query')))

검색어 삼성전자 종가 유사 종목 뉴스 요약
[{'title': '이란 물가 500% 급등 전망…유로존 PMI도 위축 국면 전환', 'content': '맞춤형 뉴스 추천 및 요약 서비스’입니다. 독자 유형별 맞춤 뉴스 6개를 선별해 제공합니다 ■ 이란... ▶ 기사 바로가기: 이란 물가 500% 폭등 부메랑, 유럽·中 거쳐 美경제 때린다 ▶ 기사 바로가기: 삼성전자... ', 'date': '2026-04-25', 'link': 'https://n.news.naver.com/mnews/article/011/0004614559?sid=101'}, {'title': '미국 연준 FOMC 금리인하 전면 수정... 뉴욕증시 비트코인 "이란 사태 충...', 'content': '삼성전자[005930]는 2.57% 오른 14만3천900원에 장을 마치며 종가 기준 역대 최고가를 새로 썼다. SK하이닉스[000660]는 0.94% 상승한 74만9천원으로 거래를 마감했다. 두 종목은 장 초반 미국 기술주 약세 여파로 동반... ', 'date': '2026-01-26', 'link': 'https://www.g-enews.com/view.php?ud=202601152006275515906806b77b_1'}, {'title': '[DD퇴근길] 코스피 장중 5000선 돌파…삼성전자, 시총 1000조', 'content': '있도록 요약했습니다. 전체 기사는 ‘디지털데일리 기사 하단의 주요뉴스(아웃링크)’에서 확인할 수... 종목 별로 살펴보면, 코스피 대장주인 삼성전자는 전장보다 1.87% 올라 지수 상승을 견인했습니다. SK하이닉스(2.... ', 'date': '2026-01-22', 'link': 'https://n.news.naver.com/mnews/article/138/0002215946?sid=101'}, {'title': '이란 물가 500% 급등 전망…유로존 PMI도 위축 국면 전환', 'content': '맞춤형 뉴스 추천 및 요약 서비스’입니다. 독자 유형